# openCypher reference

This notebook is a reference for the openCypher used in this workshop, and every example runs against
the workshop's evidence graph in Neptune Analytics. It's worth keeping open in a second tab, and you
can run any cell to see what it returns. It doesn't depend on the other notebooks: the examples use
past decisions from the history, which are the same in every account.

**How the cells work.** Each example is a `%%oc` cell, which runs the query in Neptune Analytics and
shows the result. Choose the **Table** tab for rows, or the **Graph** tab when the query returns a
path (`RETURN p`). `-d` picks the property each kind of node is labeled with in the graph view.

**Values come in as parameters.** The examples never paste a value into the query. They write
`$decision` and the like, and `-qp CHEAT` passes the values from the dict in the first cell. That's
the practice to follow in your own code: the query text stays the same for any value, and no value can
change what the query means. Change a value in the first cell and run an example again.

**Anchor every query.** This graph holds 3 million decisions. A query that starts from a known node,
such as ``{`~id`: $decision}``, reads only what's connected to it and answers in milliseconds. A query
that scans every node of a label, or every edge in the graph, can run out of memory. The examples
start from an id, or from a label with only a few nodes, such as `Version` or `Policy`.

**Reading is safe, and writing is fenced off.** The read examples change nothing. The write examples
touch only nodes labeled `CheatSheet`, and the last of them deletes them. Don't point a write at the
workshop's own labels, or the session's results will change.

| Section | What's in it |
|---|---|
| 1 · Match a pattern | nodes by id, label and property; edges in both directions; paths; optional matches; existence |
| 2 · Walk any depth | fixed and variable-length hops, the pattern behind the lineage questions |
| 3 · Return values | aliases, distinct values, ordering, counting, paging, unions |
| 4 · Aggregate, and work with lists | `WITH`, `count()`, `collect()`, list comprehensions |
| 5 · Subqueries | `CALL { }` once per row, and the nearest path |
| 6 · Neptune Analytics algorithms | breadth-first search, and vector similarity from a node |
| 7 · Parameters | the same query, other values |
| 8 · Write, safely | `CREATE`, `MERGE`, `SET`, `REMOVE`, `DELETE` on a scratch label |
| 9 · Operators and functions | the ones used in this workshop |

The data model, for reference. What decisions share:
`(Application)-[:HAS_DECISION]->(Decision)`, and from the decision `RETRIEVED_POLICY {issued, version}`,
`CITED_POLICY {reference}` and `VIOLATED` to `Policy`, `READ` to `Chunk` (the passages it was given),
`USED_TOOL` to `Tool`, and `IN_BAND` to `Band`. The lineage: `(Decision)-[:USED]->(Artifact)`, each
artifact `DERIVED_FROM` (amounts) or `IDENTITY_FROM` (an employer's identity) what it was built from,
back to a `FeedBatch` or to an extraction `PRODUCED_BY` a `Version`; and `(Notice)-[:FOR]->(Decision)`,
with `RESTS_ON` to the artifacts the notice relied on.

In [ ]:
# MIT No Attribution
#
# Copyright 2026 Amazon.com, Inc. or its affiliates. All Rights Reserved.
#
# Permission is hereby granted, free of charge, to any person obtaining a copy of this
# software and associated documentation files (the "Software"), to deal in the Software
# without restriction, including without limitation the rights to use, copy, modify,
# merge, publish, distribute, sublicense, and/or sell copies of the Software, and to
# permit persons to whom the Software is furnished to do so.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED,
# INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A
# PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT
# HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION
# OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
# SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
#
# SPDX-License-Identifier: MIT-0
# Connect the notebook to the graph, and set the values every example below uses (-qp CHEAT)
from agent.session import connect
connect(globals(), graph=True)

CHEAT = {
    "decision": "decision:hist-1346840_decision",       # a past denial that sent a notice
    "application": "HIST-1346840",
    "decisions": ["decision:hist-1346840_decision", "decision:hist-2132043_decision",
                  "decision:hist-0495353_decision"],
    "policy": "policy:HUD 4000.1 II.A.5.d",             # the qualifying-ratio rule
    "source": "version:doc-extract:2.3.1",              # the model version Decision 3 audits
    "record": "art:employer_record:E000676:2026-05-06",  # the employer identity 2.3.1 read from a pay stub
    "passage": "HUD 4000.1 II.A.5.d: Final Underwriting Decision",  # the first passage of II.A.5.d
}

## 1 · Match a pattern

`MATCH` describes a shape, with nodes in parentheses and edges in square brackets, and an arrow for
the direction. Neptune returns every place in the graph where the shape fits.

#### Nodes with a label

`Version` has only 16 nodes, so matching the whole label is cheap.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (v:Version) RETURN v.name, v.live_from ORDER BY v.name

#### A node by its id

`~id` is the node's id. Matching on it is a direct lookup, which is why the workshop's queries start
from one.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (d:Decision {`~id`: $decision}) RETURN d.application_id, d.outcome, d.day

#### A node by a property, with WHERE

The property map in a pattern is shorthand for a `WHERE` clause. `Policy` is a small label, so either
form is cheap here; on `Decision`, start from an id instead.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (p:Policy) WHERE p.id STARTS WITH 'HUD 4000.1 II.A.5' RETURN p.id ORDER BY p.id

#### Follow an edge out, and read its properties

`-[r:RETRIEVED_POLICY]->` names the edge `r`, so we can read the edition the decision read.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (d:Decision {`~id`: $decision})-[r:RETRIEVED_POLICY]->(p:Policy)
RETURN p.id, r.issued, r.version ORDER BY p.id

#### Follow an edge in

The arrow points the other way, so this finds the decisions that failed the policy. Counting them is
cheap, because the walk starts from one `Policy` node.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (p:Policy {`~id`: $policy})<-[:VIOLATED]-(d:Decision)
RETURN count(d) AS decisions_that_failed_it

#### Return a path, and draw it

Choose the **Graph** tab. `|` matches any of the edge types listed.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH p = (:Application)-[:HAS_DECISION]->(d:Decision {`~id`: $decision})
          -[:RETRIEVED_POLICY|CITED_POLICY|VIOLATED]->(:Policy)
RETURN p

#### An optional match

`OPTIONAL MATCH` keeps the row when the pattern is missing, and the missing pieces come back as
`null`, the way a SQL `LEFT JOIN` does. `id(d) IN $decisions` starts from a list of ids.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (d:Decision) WHERE id(d) IN $decisions
OPTIONAL MATCH (n:Notice)-[:FOR]->(d)
RETURN d.application_id, d.outcome, count(n) AS notices ORDER BY d.application_id

#### Whether a pattern exists, or doesn't

A pattern in `WHERE` tests for existence without returning anything. `WHERE NOT` tests for absence,
which is how Decision 4 finds a citation with no retrieval behind it.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (d:Decision {`~id`: $decision})-[c:CITED_POLICY]->(p:Policy)
WHERE NOT (d)-[:RETRIEVED_POLICY]->(p)
RETURN p.id AS cited_but_never_retrieved, c.reference AS cited_as

## 2 · Walk any depth

A `*` on an edge repeats it, so the query says what to follow rather than how deep the trail is.
openCypher never follows the same edge twice in one path, so a walk always ends. Give it an upper
bound, such as `*1..6`, chosen from the depth of your own data: here, an employer record is refreshed
once a month, so six hops cover half a year of refreshes.

#### Exactly two hops

The record's refreshes are `employer_record` artifacts that took their identity from it; two hops
back reaches the second refresh, or an income built from the first one.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (:Artifact {`~id`: $record})<-[:IDENTITY_FROM*2]-(a:Artifact)
RETURN a.kind, count(*) AS n ORDER BY a.kind

#### Between one and six hops, with the length of each path

`length(p)` is the number of edges in a path.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH p = (:Artifact {`~id`: $record})<-[:IDENTITY_FROM*1..6]-(:Artifact {kind: 'income'})<-[:USED]-(d:Decision)
RETURN d.application_id, d.day, length(p) AS hops ORDER BY d.day LIMIT 10

#### The same walk, drawn

Notebook 01's Cell 3.1 draws this walk with the version on the left; here it uses the notebook's
usual left-to-right layout, so 2.3.1 is on the right. Choose the **Graph** tab.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH p = (:Artifact {`~id`: $record})<-[:IDENTITY_FROM*1..6]-(:Artifact {kind: 'income'})<-[:USED]-(:Decision)
RETURN p LIMIT 8

## 3 · Return values

`RETURN` says what comes back. Everything after it works the way it does in SQL.

#### A property, with an alias

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (v:Version) RETURN v.component AS component, v.version AS version ORDER BY component, version

#### Distinct values

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (v:Version) RETURN DISTINCT v.component AS component ORDER BY component

#### Ordered, ascending and descending

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (v:Version) RETURN v.name, v.live_from ORDER BY v.live_from DESC, v.name ASC

#### Count

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (p:Policy) RETURN count(p) AS policies

#### Limit, and page

`SKIP 5 LIMIT 5` is the second page of five.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (v:Version) RETURN v.name ORDER BY v.name SKIP 5 LIMIT 5

#### Combine two queries: UNION and UNION ALL

`UNION` drops duplicate rows, and `UNION ALL` keeps them. Both parts must return the same columns.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (:Decision {`~id`: $decision})-[:RETRIEVED_POLICY]->(p:Policy) RETURN p.id AS policy, 'retrieved' AS how
UNION
MATCH (:Decision {`~id`: $decision})-[:CITED_POLICY]->(p:Policy) RETURN p.id AS policy, 'cited' AS how

## 4 · Aggregate, and work with lists

`WITH` passes rows from one part of a query to the next, and it's where we aggregate, filter the
aggregate, or sort before going on.

#### Group, then filter the group

Each policy, with every kind of edge this decision has to it, kept only when there's more than one.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (:Decision {`~id`: $decision})-[r]->(p:Policy)
WITH p, collect(type(r)) AS edges
WHERE size(edges) > 1
RETURN p.id, edges ORDER BY p.id

#### Collect rows into a list

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (:Decision {`~id`: $decision})-[:CITED_POLICY]->(p:Policy)
RETURN count(p) AS cited, collect(p.id) AS policies

#### A path as a list of names

A list comprehension, `[x IN list | expression]`, turns each node on the path into a label.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH p = (:Version {`~id`: $source})<-[:PRODUCED_BY]-(:Artifact)<-[:IDENTITY_FROM*1..6]-(:Artifact {kind: 'income'})<-[:USED]-(:Decision)
RETURN [n IN nodes(p) | coalesce(n.name, n.kind, n.application_id)] AS trail, length(p) AS hops
ORDER BY hops DESC LIMIT 3

#### Filter a list

`[x IN list WHERE condition]` keeps the elements that match, and `size()` counts them.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH p = (:Artifact {`~id`: $record})<-[:IDENTITY_FROM*1..6]-(:Artifact {kind: 'income'})<-[:USED]-(d:Decision)
RETURN d.application_id,
       size([n IN nodes(p) WHERE n.kind = 'employer_record']) AS records_on_the_path
ORDER BY records_on_the_path DESC LIMIT 5

## 5 · Subqueries

`CALL { WITH x ... RETURN y }` runs a subquery once for each incoming row. `ORDER BY ... LIMIT`
inside it applies per row.

#### A subquery once per row

For each decision, how many policies it retrieved and how many it cited.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (d:Decision) WHERE id(d) IN $decisions
CALL {
  WITH d
  MATCH (d)-[r:RETRIEVED_POLICY|CITED_POLICY]->(:Policy)
  RETURN sum(CASE type(r) WHEN 'RETRIEVED_POLICY' THEN 1 ELSE 0 END) AS retrieved,
         sum(CASE type(r) WHEN 'CITED_POLICY' THEN 1 ELSE 0 END) AS cited
}
RETURN d.application_id, retrieved, cited ORDER BY d.application_id

#### The nearest path

Order the paths by length and keep the first: the shortest trail from the version to a decision.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH p = (:Version {`~id`: $source})<-[:PRODUCED_BY]-(:Artifact)<-[:IDENTITY_FROM*1..6]-(:Artifact {kind: 'income'})<-[:USED]-(d:Decision)
RETURN d.application_id, d.day, length(p) AS hops ORDER BY hops, d.day LIMIT 1

## 6 · Neptune Analytics algorithms

Neptune Analytics runs graph algorithms inside a query, with `CALL neptune.algo...` and `YIELD`.

#### Breadth-first search

`bfs.levels` starts at a node, follows the listed edges (inbound here), visits each node once, and
says how many hops away it is. This is notebook 01's Cell 3.2 walk, counted by level.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (v:Version {`~id`: $source})
CALL neptune.algo.bfs.levels(v, {traversalDirection: 'inbound',
     edgeLabels: ['PRODUCED_BY', 'DERIVED_FROM', 'IDENTITY_FROM', 'USED', 'RESTS_ON']})
YIELD node, level
WHERE 'Decision' IN labels(node)
RETURN level, count(node) AS decisions ORDER BY level

#### Vector similarity, from a node

Each passage the Knowledge Base holds is a `Chunk` node with its embedding. `vectors.topK.byNode`
finds the passages nearest to one of them: here, the first passage of HUD 4000.1 II.A.5.d. Lower
scores are closer, and the nearest is the passage itself. (`Chunk` has under 200 nodes, so finding it
by its text is cheap. Live runs also link each decision to the passages it read, with `READ`.)

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (c:Chunk) WHERE c.text CONTAINS $passage
WITH c LIMIT 1
CALL neptune.algo.vectors.topK.byNode(c, {topK: 4})
YIELD node, score
RETURN id(node) AS passage, score, left(node.text, 80) AS starts ORDER BY score

## 7 · Parameters

Every example in this notebook already uses parameters. Here's what they buy: the same query text
runs for any value, so it's safe to build into code, such as the MCP tools in notebook 03.

In [ ]:
# A Python cell: the same parameters, for another past decision
OTHER = dict(CHEAT, decision="decision:hist-2132043_decision")

#### The same query, other values

In [ ]:
%%oc -qp OTHER
MATCH (d:Decision {`~id`: $decision})-[:CITED_POLICY]->(p:Policy)
RETURN d.application_id, collect(p.id) AS cited

## 8 · Write, safely

These cells touch only nodes labeled `CheatSheet`, and the last one deletes them all. Run them in
order. In the workshop, only the stream transform writes the graph, and it uses `MERGE`, so a record
delivered twice adds nothing.

#### Create nodes, and an edge between them

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
CREATE (a:CheatSheet {name: 'demo-a'})-[:LINKS_TO {weight: 1}]->(b:CheatSheet {name: 'demo-b'})
RETURN a.name, b.name

#### Create several from a list

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
UNWIND [{name: 'demo-c'}, {name: 'demo-d'}] AS props
CREATE (n:CheatSheet) SET n = props
RETURN n.name

#### Merge: match it, or create it

`demo-a` already exists, so this matches it and increments `seen`. Run it again to watch the count go
up. This is how the stream transform makes a replay safe.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MERGE (n:CheatSheet {name: 'demo-a'})
  ON CREATE SET n.created = true
  ON MATCH SET n.seen = coalesce(n.seen, 0) + 1
RETURN n.name, n.seen, n.created

#### Set, and remove, a property

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (n:CheatSheet {name: 'demo-b'})
SET n.colour = 'blue'
REMOVE n.colour
RETURN n.name, n.colour

#### Delete an edge

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (:CheatSheet)-[r:LINKS_TO]->(:CheatSheet)
DELETE r
RETURN count(r) AS edges_deleted

#### Delete the scratch nodes, and any edges they still have

`DETACH DELETE` removes a node with its edges; a plain `DELETE` fails on a node that still has edges.

In [ ]:
%%oc -qp CHEAT -d {"Decision":"outcome","Policy":"id","Artifact":"kind","Version":"name","Notice":"day","Application":"application_id"} -l 30
MATCH (n:CheatSheet)
DETACH DELETE n
RETURN count(n) AS nodes_deleted

## 9 · Operators and functions

| Type | Operators |
|---|---|
| General | `DISTINCT`, `x.y` (property access), `|` between edge types |
| Math | `+`, `-`, `*`, `/`, `%` |
| Comparison | `=`, `<>`, `<`, `>`, `<=`, `>=`, `IS NULL`, `IS NOT NULL` |
| Boolean | `AND`, `OR`, `NOT`, `XOR` |
| String | `STARTS WITH`, `ENDS WITH`, `CONTAINS`, `+` (concatenate) |
| List | `+` (concatenate), `IN`, `[i]` (index), `[i..j]` (slice) |

| Type | Functions |
|---|---|
| Scalar | `coalesce()`, `id()`, `type()`, `labels()`, `size()`, `length()`, `head()`, `toString()`, `toInteger()` |
| Aggregate | `count()`, `sum()`, `avg()`, `min()`, `max()`, `collect()` |
| List | `nodes()`, `relationships()`, `range()`, `tail()`, `keys()` |
| String | `left()`, `toUpper()`, `toLower()`, `trim()`, `replace()`, `split()` |
| Math | `round()`, `abs()`, `sqrt()` |

The cell below runs each of them on one decision.

In [ ]:
%%oc -qp CHEAT
MATCH p = (d:Decision {`~id`: $decision})-[r:VIOLATED]->(pol:Policy)
RETURN coalesce(d.missing, 'default') AS coalesce_, size(pol.id) AS size_, type(r) AS type_,
       head(labels(d)) AS head_labels_, toString(d.compensating_factors) + ' factors' AS toString_,
       toInteger('42') AS toInteger_, length(p) AS length_, keys(d)[0..2] AS keys_,
       range(1, 3) AS range_, tail([1, 2, 3]) AS tail_, left(pol.id, 10) AS left_,
       toUpper('deny') AS toUpper_, replace(pol.id, 'HUD ', '') AS replace_, split('a,b', ',') AS split_,
       round(2.567) AS round_, sqrt(16) AS sqrt_, abs(-3) AS abs_
LIMIT 1